# 2일차 예제 노트북 · LLM API · 구조화 출력 · RAG 의 원리

강의안의 예제 코드가 모두 이 노트북 하나에 있습니다. 셀 제목의 번호(예제 1-1 (1))는 강의안 코드 장 제목의 번호와 같습니다.
위에서부터 순서대로 실행합니다. 파트 1 · 2 는 문의 120건을 분류해 엑셀과 DB 에 저장하고, 파트 3 은 사내 문서 253건을 검색합니다.
실습 미션은 따로 받은 `2일차_실습.ipynb` 에서 합니다.

**처음 한 번만 할 일 · 코랩**
1. [파일] → [드라이브에 사본 저장] 을 누르고, 새 탭에 열린 사본에서 진행합니다.
2. 왼쪽 열쇠 아이콘(보안 비밀) → 「새 보안 비밀 추가」 → 이름 `ANTHROPIC_API_KEY`, 값에 받은 키를 붙여 넣고 「노트북 액세스」를 켭니다.
3. 아래 「환경 준비」 셀 세 개를 차례로 실행합니다(Shift + Enter). (1) 재료 · 패키지 → (2) 키 → (3) 연결 확인 순서입니다. (2) 셀에서 「데이터 접근」 허용 창이 뜨면 [허용]을 누릅니다. 마지막 셀에 `준비 완료` 가 나오면 끝입니다.

**막히거나 더 알고 싶을 때** PC 의 Claude Code 에 셀을 붙여 넣고 물어봅니다.
1. 「이 줄이 하는 일을 한 문장으로 알려 줘」
2. 「이 값을 바꿔 실행하면 무엇이 달라지는지 보여 줘」
3. 「이 오류가 왜 났고 어떻게 고치는지 알려 줘」

코드 셀마다 맨 아래 주석에 그 셀에 맞는 질문을 하나씩 적어 두었습니다.

## 환경 준비

실습 재료 ZIP 을 학습 사이트에서 내려받아 현재 폴더에 풀고, 코랩에 없는 패키지를 설치합니다. 파일을 따로 올리거나 압축을 풀 필요가 없습니다.
- `anthropic` 은 Claude API 를 호출하는 공식 파이썬 SDK(개발 도구 묶음)이고, `openpyxl` 은 pandas 가 엑셀 파일을 쓸 때 사용하는 패키지, `sentence-transformers` 는 파트 3 에서 문장을 벡터(임베딩)로 바꾸는 라이브러리입니다.
- `%pip` 는 노트북 안에서 pip 설치를 실행하는 명령입니다. `-q` 는 설치 로그를 줄입니다.
- `inquiries_120.csv` 가 이미 있으면 재료를 다시 받지 않습니다.

설치 뒤 「you may need to restart the kernel」 안내가 떠도 재시작하지 않고 다음 셀로 갑니다.

출력된 CSV 파일 목록에 `inquiries_120.csv` 가 있으면 됩니다.

In [ ]:
# 환경 준비 (1) 실습 재료를 내려받고 패키지를 설치합니다 · 처음 한 번만
%pip install -q anthropic openpyxl sentence-transformers
import os
import urllib.request
import zipfile

URL = 'https://lecture-web-five.vercel.app/materials/day02-data.zip'
# 재료가 이미 있으면 다시 받지 않습니다
if not os.path.exists('inquiries_120.csv'):
    urllib.request.urlretrieve(URL, 'day02-data.zip')
    zipfile.ZipFile('day02-data.zip').extractall('.')
# 받은 CSV 파일 목록을 확인합니다
csv_files = [file_name for file_name in os.listdir('.') if file_name.endswith('.csv')]
print(sorted(csv_files))

코랩 보안 비밀에 저장한 API 키를 읽어 환경 변수 `ANTHROPIC_API_KEY` 에 넣습니다. 보안 비밀은 환경 변수에 저절로 들어가지 않으므로, 이 셀을 건너뛰면 다음 셀에서 「Could not resolve authentication method」 오류가 납니다. 키를 코드에 직접 적으면 노트북을 공유할 때 키도 함께 퍼집니다.
- **환경 변수**: 프로그램이 실행 중에 읽는 이름과 값의 쌍입니다. SDK 는 이 이름의 값을 키로 씁니다.
- **try / except**: `try` 안에서 오류(예외)가 나면 셀을 멈추지 않고 `except` 아래를 실행합니다. 예외 처리는 예제 1-3 에서 자세히 다룹니다.
- 처음 실행하면 「데이터 접근」 허용 창이 뜹니다. [허용]을 누릅니다.

출력에서 키 앞부분이 `sk-ant-` 로 시작하는지 확인합니다. 키를 읽지 못하면 무엇을 확인할지 세 줄이 출력됩니다.

In [ ]:
# 환경 준비 (2) 보안 비밀의 키를 환경 변수로 옮깁니다 · 키를 코드에 적지 않습니다
import os
from google.colab import userdata                 # 코랩 보안 비밀

try:
    api_key = userdata.get('ANTHROPIC_API_KEY').strip()
    os.environ['ANTHROPIC_API_KEY'] = api_key
    print('키 앞부분', api_key[:7], '· 길이', len(api_key))
except Exception as error:
    print('보안 비밀에서 키를 읽지 못했습니다:', type(error).__name__)
    print('1. 열쇠 아이콘의 이름이 ANTHROPIC_API_KEY 와 같은지 봅니다')
    print('2. 그 줄의 「노트북 액세스」가 켜져 있는지 봅니다')
    print('3. 「데이터 접근」 창이 떴다면 [허용]을 누릅니다')
    print('확인한 뒤 이 셀을 다시 실행합니다')

API 를 호출할 클라이언트를 만들고, 오늘 쓸 모델 이름을 변수에 담은 뒤, 경량 모델에 짧은 요청을 한 번 보내 연결을 확인합니다.
- **client**: API 서버와 통신하는 객체입니다. 이후 모든 호출은 `client.messages.create(...)` 형태로 실행합니다.
- **모델 등급**: 경량(haiku)은 빠르고 저렴하고, 대형(opus)은 느리고 비쌉니다. 중형(sonnet)은 그 사이입니다.
- **messages.create**: 모델에 요청을 보내는 함수입니다. `model`, `max_tokens`, `messages` 를 넣으며, 각 인자의 뜻은 예제 1-1 에서 하나씩 봅니다.

SDK 버전과 `준비 완료` 가 나오면 준비가 끝났습니다.

In [ ]:
# 환경 준비 (3) 클라이언트와 모델 이름을 정하고 연결을 확인합니다
import anthropic
import json
import time
import pandas as pd

client = anthropic.Anthropic()        # 환경 변수 ANTHROPIC_API_KEY 를 읽습니다
MODEL       = 'claude-sonnet-5-5'     # 중형 · 오늘의 기본 모델
SMALL_MODEL = 'claude-haiku-4-5'      # 경량 · 대량 분류용
BIG_MODEL   = 'claude-opus-5-5'       # 대형 · 비교용
print('anthropic', anthropic.__version__)

# 경량 모델에 짧은 요청을 보내 연결을 확인합니다
response = client.messages.create(
    model=SMALL_MODEL,
    max_tokens=20,
    messages=[{'role': 'user', 'content': '"준비 완료" 라고만 답해줘.'}])
# 경량 모델은 답 블록 하나만 돌려줍니다
print(response.content[0].text)

## 예제 1-1 · 첫 호출과 대화 이력
문의 한 건을 코드로 보내 응답 객체를 열어 보고, 대화 이력을 코드로 쌓아 토큰이 어떻게 늘어나는지 확인합니다.

문의 한 건을 요약해 달라는 요청을 중형 모델에 보냅니다.
- **messages**: 대화 목록입니다. 항목마다 `role`(말한 쪽: `user` 는 사람, `assistant` 는 모델)과 `content`(내용)를 넣습니다.
- **max_tokens**: 응답 길이 상한입니다. 토큰은 모델이 글을 잘라 처리하는 단위입니다.
- **text_of()**: 응답 본문은 블록 목록입니다. 답 글자(`text`) 외의 블록도 섞일 수 있어 `text` 블록만 골라 잇습니다. 이후 예제에서 계속 씁니다.

출력에서 요약 한 문장을 확인합니다.

In [ ]:
# 예제 1-1 (1) 한 번 호출해 봅니다
def text_of(response):
    """응답 블록 가운데 text 블록만 이어 붙여 답 글자를 돌려줍니다."""
    return ''.join(block.text for block in response.content if block.type == 'text')

question = ('3호기 로드락 도어 인터락 알람이 세 번 떴습니다. '
            '이 문의를 한 문장으로 요약해줘.')
# role 은 말한 쪽, content 는 내용입니다
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,                          # 응답 길이 상한(토큰)
    messages=[{'role': 'user', 'content': question}])
print(text_of(response))

# Claude Code 에 물어보기 ·
#   「question 을 영어로 바꿔 보내고 출력 토큰 차이를 알려 줘」

방금 받은 응답 객체 `response` 의 속성을 하나씩 출력합니다.
- **content**: 응답 블록 목록입니다. 모델이 답 전에 거친 중간 단계인 `thinking` 블록이 먼저 올 수 있고, 답인 `text` 블록이 뒤에 옵니다. `text_of()` 는 `text` 블록만 모아 돌려줍니다.
- **usage**: 이번 호출의 입력 토큰 수와 출력 토큰 수입니다. 요금은 이 두 숫자로 계산합니다.
- **stop_reason**: 응답이 끝난 이유입니다. `end_turn` 은 정상 종료이고, `max_tokens` 는 길이 상한에서 잘린 경우입니다.

출력에서 블록 종류, 입력·출력 토큰 수, 종료 이유 `end_turn` 을 확인합니다.

In [ ]:
# 예제 1-1 (2) 응답 객체에 무엇이 들어 있는지 봅니다
block_types = [block.type for block in response.content]
print('블록 종류', block_types)
print('모델    ', response.model)
print('입력 토큰', response.usage.input_tokens)
print('출력 토큰', response.usage.output_tokens)
print('종료 이유', response.stop_reason)

# Claude Code 에 물어보기 ·
#   「thinking 블록 안의 글도 함께 찍도록 고쳐 줘」

같은 질문을 두 번 보내 답을 비교합니다. 모델은 매번 답을 새로 생성하므로 같은 질문에도 문장이 달라질 수 있습니다.

출력에서 두 답의 표현이 어디서 달라지는지 봅니다. 이 차이는 예제 2-3 (3) 에서 두 번 돌려 답이 갈린 건을 찾을 때 다시 다룹니다.

In [ ]:
# 예제 1-1 (3) 같은 질문을 두 번 보냅니다
for i in range(2):
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{'role': 'user', 'content': question}])
    print(i + 1, text_of(response))

# Claude Code 에 물어보기 ·
#   「다섯 번 보내서 답이 완전히 같은 쌍이 있는지 세어 줘」

분류 규칙은 `system` 에, 분류할 문의는 `messages` 에 나눠 넣습니다.
- **system**: 대화 전체에 적용할 지시문입니다. 요청마다 같은 규칙을 넣을 때 씁니다.
- **JSON**: `{"키": 값}` 모양으로 데이터를 적는 텍스트 형식입니다. 파이썬 딕셔너리와 모양이 비슷합니다.

출력에서 JSON 한 줄이 돌아오는지, 앞뒤에 다른 글이 붙지 않았는지 봅니다.

In [ ]:
# 예제 1-1 (4) 규칙은 system 에, 요청은 messages 에 나눠 넣습니다
SYS_CLS = ('너는 반도체 공장 현장 문의를 분류하는 도우미다. '
           '[문서안내/절차안내/긴급/데이터요청] 네 가지 중 하나로만 분류한다. '
           '출력은 {"id":..., "category":..., "urgent":true/false} JSON 한 줄만.')
body = 'id: Q101\n문의: 3호기 로드락 도어 인터락 알람이 세 번 떴습니다'
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=SYS_CLS,                  # 모든 요청에 공통으로 적용할 규칙
    messages=[{'role': 'user', 'content': body}])
print(text_of(response))

# Claude Code 에 물어보기 ·
#   「네 라벨에 각각 한 줄 설명을 더한 SYS_CLS 를 써 줘」

앞의 질문과 답을 `messages` 에 이어 붙이고 새 질문을 덧붙여 보냅니다. API 는 이전 대화를 저장하지 않으므로 이력을 매번 직접 보냅니다.
- `role` 이 `assistant` 인 항목에는 직전 응답을 넣습니다. 목록은 `user` 와 `assistant` 가 번갈아 오고 `user` 로 끝나야 합니다.

출력에서 입력 토큰이 앞 호출보다 늘어난 것을 확인합니다.

In [ ]:
# 예제 1-1 (5) 이전 대화를 messages 에 이어 붙여 다음 질문을 보냅니다
next_body = 'id: Q102\n문의: PM 체크리스트 최신본 위치를 알고 싶습니다'
history = [{'role': 'user',      'content': body},
           {'role': 'assistant', 'content': text_of(response)},   # 직전 응답을 넣습니다
           {'role': 'user',      'content': next_body}]
second_response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=SYS_CLS,
    messages=history)
print(text_of(second_response))
print('입력 토큰', response.usage.input_tokens,
      '→', second_response.usage.input_tokens)

# Claude Code 에 물어보기 ·
#   「history 가 user 로 끝나지 않으면 무슨 오류가 나는지 보여 줘」

질문 네 개를 차례로 보내면서 질문과 답을 `history` 에 계속 쌓습니다.
- `enumerate(inquiry_texts, 1)` 은 목록을 돌면서 1부터 시작하는 번호를 함께 돌려줍니다.

출력에서 호출할 때마다 입력 토큰이 커지는 것을 봅니다. 이력이 길수록 요금도 늘어납니다.

In [ ]:
# 예제 1-1 (6) 이력이 쌓이면 입력 토큰도 쌓입니다
history = []
inquiry_texts = ['5호기 챔버 압력 알람이 지속됩니다',
                 'PM 체크리스트 최신본 위치를 알고 싶습니다',
                 '지난주 2라인 파티클 측정값을 받을 수 있나요',
                 '클린룸 출입 교육 일정이 궁금합니다']
for i, inquiry_text in enumerate(inquiry_texts, 1):
    # 질문을 이력에 넣고 이력 전체를 보냅니다
    history.append({'role': 'user',
                    'content': f'id: Q{200 + i}\n문의: {inquiry_text}'})
    response = client.messages.create(
        model=SMALL_MODEL,
        max_tokens=200,
        system=SYS_CLS,
        messages=history)
    # 답도 이력에 쌓습니다
    history.append({'role': 'assistant', 'content': text_of(response)})
    print(f'{i}번째 호출 · 입력 토큰 {response.usage.input_tokens}')

# Claude Code 에 물어보기 ·
#   「10턴까지 늘리면 입력 토큰 합계가 얼마인지 계산해 줘」

이력 전체 대신 최근 한 턴과 마지막 질문만 잘라 보냅니다.
- **리스트 슬라이싱** `history[-4:-1]`: 뒤에서 네 번째 항목부터 마지막 항목 바로 앞까지 꺼냅니다. 음수 번호는 뒤에서 셉니다.

출력에서 입력 토큰이 예제 1-1 (6) 의 마지막 호출보다 줄었는지 봅니다. 답의 category 가 네 라벨 중 하나인지, 앞뒤에 다른 글자가 붙었는지도 봅니다.

In [ ]:
# 예제 1-1 (7) 이력은 최근 것만 남겨 보냅니다 · 직전 한 턴 + 마지막 질문
# history 는 [질문1, 답1, …, 질문4, 답4] 입니다. 마지막 답을 빼야 질문으로 끝납니다
recent = history[-4:-1]        # [질문3, 답3, 질문4]
response = client.messages.create(
    model=SMALL_MODEL,
    max_tokens=200,
    system=SYS_CLS,
    messages=recent)
print('전부 보냈을 때와 비교 · 최근 것만 보낸 입력 토큰',
      response.usage.input_tokens)
print(text_of(response))

# Claude Code 에 물어보기 ·
#   「마지막 질문 하나만 보내는 슬라이스로 고쳐 돌려 줘」

응답 길이 상한과 추론 정도를 바꿔 결과를 비교합니다.
- `max_tokens=10` 처럼 상한이 너무 작으면 `stop_reason` 이 `max_tokens` 가 됩니다. thinking 블록이 상한을 먼저 쓰면 text 는 빈 문자열 `''` 로 옵니다. 출력의 블록 종류에서 확인합니다.
- **output_config**: 출력 설정을 담는 인자입니다. 여기서는 `effort`(추론에 들이는 정도, `low`·`high`)를 넣고, 예제 2-1 (3) 에서는 응답 형식(스키마)을 넣습니다.

출력에서 effort 에 따라 걸린 시간, 출력 토큰, 블록 종류가 어떻게 다른지 봅니다.

In [ ]:
# 예제 1-1 (8) 파라미터 둘을 바꿔 봅니다 · max_tokens 와 effort
# 상한이 너무 작으면 답이 잘립니다
response = client.messages.create(
    model=MODEL,
    max_tokens=10,
    messages=[{'role': 'user', 'content': question}])
block_types = [block.type for block in response.content]
print('max_tokens=10 →', response.stop_reason, block_types, repr(text_of(response)))

# effort 를 바꿔 걸린 시간 · 출력 토큰 · 블록 종류를 비교합니다
for effort in ['low', 'high']:
    started = time.time()
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        output_config={'effort': effort},    # 추론 정도 · low 는 빠르고 짧게
        messages=[{'role': 'user', 'content': question}])
    seconds = time.time() - started
    block_types = [block.type for block in response.content]
    output_tokens = response.usage.output_tokens
    print(f'effort={effort:4s} {seconds:4.1f}초 · 출력 {output_tokens}'
          f' · {block_types}')

# Claude Code 에 물어보기 ·
#   「effort 가 답 내용에 영향을 줬는지 두 답을 비교해 줘」

## 예제 1-2 · 토큰 계산과 모델 선택
보내기 전에 토큰을 세서 120건 요금을 미리 계산하고, 같은 10건을 세 등급 모델로 실제로 돌려 고르는 근거를 만듭니다.

요청을 보내기 전에 입력 토큰을 세고, 실제로 보낸 뒤의 `usage` 와 비교합니다.
- **count_tokens**: `create` 와 같은 `model` · `system` · `messages` 를 받아 입력 토큰 수만 돌려주는 함수입니다. `max_tokens` 는 넣지 않습니다. 모델이 답을 생성하지 않아 요금이 들지 않습니다.

출력에서 두 숫자가 같은지 확인합니다.

In [ ]:
# 예제 1-2 (1) 보내기 전에 토큰을 셉니다 · count_tokens 는 요금이 들지 않습니다
# 같은 system · messages 로 먼저 세고, 그다음 실제로 보냅니다
token_count = client.messages.count_tokens(
    model=MODEL,
    system=SYS_CLS,
    messages=[{'role': 'user', 'content': body}])
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=SYS_CLS,
    messages=[{'role': 'user', 'content': body}])
print('보내기 전 계산', token_count.input_tokens)
print('보낸 뒤 usage ', response.usage.input_tokens)

# Claude Code 에 물어보기 ·
#   「count_tokens 로 system 유무의 토큰 차이를 재 줘」

문의 120건마다 입력 토큰을 세어 `in_tok` 열에 넣습니다. 빈 문의는 보내지 않으므로 0 으로 둡니다.
- **read_csv · fillna('')**: CSV 파일을 pandas 표(DataFrame)로 읽고, 빈 칸을 빈 문자열로 바꿉니다.
- **ThreadPoolExecutor**: 여러 건을 동시에 처리하는 파이썬 표준 도구입니다. `executor.map(함수, 목록)` 은 목록의 항목마다 함수를 적용하되 `max_workers` 개씩 동시에 실행합니다. 한 건씩 처리할 때와의 시간 비교는 예제 1-2 (6) 에서 자세히 봅니다.

출력에서 빈 문의 수와 입력 토큰의 합계·평균·최대를 확인합니다.

In [ ]:
# 예제 1-2 (2) 문의 120건의 입력 토큰을 셉니다 · 빈 문의는 0 으로 둡니다
from concurrent.futures import ThreadPoolExecutor
inquiries = pd.read_csv('inquiries_120.csv').fillna('')

def count_in(text):
    """문의 글자를 받아 입력 토큰 수를 돌려줍니다. 빈 문의는 0 입니다."""
    if not text.strip():
        return 0
    counted = client.messages.count_tokens(
        model=SMALL_MODEL,
        system=SYS_CLS,
        messages=[{'role': 'user', 'content': text}])
    return counted.input_tokens

# 4건씩 동시에 실행합니다 · 시간 비교는 심화 예제 1-2 (6)
with ThreadPoolExecutor(max_workers=4) as executor:
    inquiries['in_tok'] = list(executor.map(count_in, inquiries['문의내용']))

# 보낸 문의(토큰 > 0)만 골라 통계를 냅니다
sent = inquiries[inquiries['in_tok'] > 0]
print('건수', len(inquiries), '· 빈 문의', len(inquiries) - len(sent))
print('입력 합계', sent['in_tok'].sum(),
      '· 평균', round(sent['in_tok'].mean(), 1),
      '· 최대', sent['in_tok'].max())

# Claude Code 에 물어보기 ·
#   「in_tok 이 가장 큰 문의 3건의 내용을 보여 줘」

예제 1-2 (2) 에서 센 입력 토큰으로 모델 등급별 120건 요금을 미리 계산합니다.
- 요금은 (입력 토큰 × 입력 단가 + 출력 토큰 × 출력 단가) ÷ 1,000,000 입니다. 단가는 100만 토큰당 달러입니다.
- 입력 토큰은 경량 모델로 센 값입니다. 모델마다 토큰 수가 조금 달라서 중형 · 대형 요금은 이보다 조금 더 나옵니다.
- 출력 토큰은 아직 모르므로 한 건에 30 토큰으로 가정합니다(`OUT_TOK`).

출력에서 등급 사이의 요금 차이가 몇 배인지 봅니다.

In [ ]:
# 예제 1-2 (3) 등급별로 120건 요금을 미리 계산합니다
# 100만 토큰당 달러 (입력, 출력) · 2026-10 공개 정가
PRICE = {SMALL_MODEL: (1, 5), MODEL: (2, 10), BIG_MODEL: (4, 20)}
OUT_TOK = 30                     # 응답은 JSON 한 줄 · 예제 1-1 (4) 근처 값
input_total = inquiries['in_tok'].sum()
call_count = (inquiries['in_tok'] > 0).sum()      # 빈 문의는 보내지 않습니다
for model_name, (input_price, output_price) in PRICE.items():
    input_cost = input_total * input_price
    output_cost = call_count * OUT_TOK * output_price
    cost_usd = (input_cost + output_cost) / 1_000_000
    print(f'{model_name:18s} ${cost_usd:.4f}')

# Claude Code 에 물어보기 ·
#   「출력 토큰을 실제 평균값으로 바꿔 요금을 다시 계산해 줘」

표본 10건을 세 모델에 실제로 보내 분류 결과, 토큰 수, 걸린 시간을 모읍니다.
- `iterrows()` 는 표를 한 행씩 꺼냅니다. `row['문의내용']` 처럼 열 이름으로 값을 읽습니다.
- `json.loads` 는 JSON 글자를 파이썬 딕셔너리로 바꿉니다. 응답 앞뒤에 다른 글이 붙어도 되도록 첫 `{` 부터 마지막 `}` 까지만 잘라 읽습니다. 이 파싱 방식은 예제 2-1 (2) 에서 자세히 봅니다.

출력에서 모델마다 앞 3건의 분류를 비교합니다.

In [ ]:
# 예제 1-2 (4) 같은 10건을 세 모델로 실제로 돌려 비교합니다
from collections import namedtuple
sample = inquiries[inquiries['in_tok'] > 0].head(10)
Result = namedtuple('Result',
                    ['category', 'input_tokens', 'output_tokens', 'seconds'])

def parse_category(answer_text):
    """답 글자에서 중괄호 사이만 JSON 으로 읽어 category 값을 돌려줍니다."""
    json_text = answer_text[answer_text.find('{'):answer_text.rfind('}') + 1]
    return json.loads(json_text)['category']

def run(model, row):
    """문의 한 행을 model 로 분류해 Result 네 값을 돌려줍니다."""
    prompt = f"id: {row['id']}\n문의: {row['문의내용']}"
    started = time.time()
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system=SYS_CLS,
        messages=[{'role': 'user', 'content': prompt}])
    seconds = time.time() - started
    category = parse_category(text_of(response))
    return Result(category, response.usage.input_tokens,
                  response.usage.output_tokens, seconds)

# 모델마다 표본 10건을 차례로 돌립니다
results_by_model = {}
for model_name in PRICE:
    results = [run(model_name, row) for _, row in sample.iterrows()]
    results_by_model[model_name] = results

# 모델별 앞 3건의 분류를 나란히 봅니다
first_three = {}
for model_name, results in results_by_model.items():
    first_three[model_name] = [result.category for result in results[:3]]
print(first_three)

# Claude Code 에 물어보기 ·
#   「세 모델 답이 다른 건의 문의 내용을 꺼내 보여 줘」

예제 1-2 (4) 의 결과를 모델별로 합쳐 걸린 시간, 요금, 대형 모델과 같은 답 수를 한 줄씩 출력합니다.
- `zip(a, b)` 는 두 목록을 같은 순서끼리 짝지어 줍니다.

출력에서 경량 모델이 대형 모델과 몇 건이나 같은 답을 냈는지, 그때 시간과 요금이 얼마나 줄었는지 봅니다. 중형 · 대형은 답을 쓰기 전의 thinking 토큰이 출력에 들어가, 출력 토큰이 경량보다 많고 예제 1-2 (3) 에서 가정한 한 건 30 토큰보다도 큽니다.

In [ ]:
# 예제 1-2 (5) 시간 · 요금 · 대형 모델과 같은 답 수를 표로 정리합니다
big_results = results_by_model[BIG_MODEL]
for model_name, results in results_by_model.items():
    seconds = sum(result.seconds for result in results)
    input_tokens = sum(result.input_tokens for result in results)
    output_tokens = sum(result.output_tokens for result in results)
    input_price, output_price = PRICE[model_name]
    cost_usd = (input_tokens * input_price + output_tokens * output_price) / 1e6
    # 대형 모델과 같은 답 수
    same_as_big = sum(result.category == big_result.category
                      for result, big_result in zip(results, big_results))
    print(f'{model_name:18s} {seconds:5.1f}초 · 입력 {input_tokens}'
          f' · 출력 {output_tokens} · ${cost_usd:.4f}'
          f' · 같은 답 {same_as_big}/10')

# Claude Code 에 물어보기 ·
#   「세 모델이 모두 일치한 건수도 함께 찍도록 고쳐 줘」

같은 10건을 한 건씩 차례로 보낼 때와 다섯 건씩 동시에 보낼 때의 시간을 잽니다.
- API 호출은 대부분의 시간을 서버 응답을 기다리는 데 씁니다. **ThreadPoolExecutor** 로 여러 호출을 동시에 실행하면 기다리는 시간이 겹쳐 전체 시간이 줄어듭니다.
- `max_workers` 는 동시에 실행할 개수입니다. 너무 크면 요청 한도(rate limit)에 걸립니다.

출력에서 두 시간을 비교합니다.

In [ ]:
# 예제 1-2 (6) 동시에 보내면 기다리는 시간이 줄어듭니다
def send_one(row):
    """문의 한 행을 경량 모델로 보내 답 글자를 돌려줍니다."""
    prompt = f"id: {row['id']}\n문의: {row['문의내용']}"
    response = client.messages.create(
        model=SMALL_MODEL,
        max_tokens=200,
        system=SYS_CLS,
        messages=[{'role': 'user', 'content': prompt}])
    return text_of(response)

sample_rows = [row for _, row in sample.iterrows()]

# 먼저 한 건씩 차례로 보냅니다
started = time.time()
answers = [send_one(row) for row in sample_rows]
sequential_seconds = time.time() - started

# 같은 10건을 다섯 건씩 동시에 보냅니다
started = time.time()
with ThreadPoolExecutor(max_workers=5) as executor:
    answers = list(executor.map(send_one, sample_rows))
parallel_seconds = time.time() - started
print(f'한 건씩 {sequential_seconds:.1f}초 · 다섯 개씩 동시에 {parallel_seconds:.1f}초')

# Claude Code 에 물어보기 ·
#   「executor.map 결과가 왜 원래 순서대로 오는지 설명해 줘」

## 예제 1-3 · 오류 처리와 재시도
오류를 일부러 일으켜 종류를 보고, 오류 처리가 없는 코드를 Claude Code 에 맡겨 설명을 듣고 고치게 한 뒤 검사 스크립트로 확인합니다.
같은 기준의 `ask()` 와 `check_response()` 를 노트북에도 두고 실제 API 로 한 번 더 확인합니다.

없는 모델, 빈 메시지, 틀린 키로 일부러 요청을 보내 어떤 오류가 나는지 봅니다.
- **예외(exception)**: 실행 중 오류가 나면 파이썬은 예외를 일으키고 셀을 멈춥니다. `try` 안의 코드에서 예외가 나면 `except` 아래로 넘어가 처리를 이어 갑니다.
- **APIStatusError**: 서버가 오류 상태 코드로 응답했을 때의 예외입니다. `status_code` 에 404(없음), 400(잘못된 요청), 401(인증 실패) 같은 숫자가 들어 있습니다.
- `max_retries=0` 은 SDK 의 자동 재시도를 끄는 설정입니다. 재시도는 예제 1-3 (2) 에서 다룹니다.

출력에서 오류 이름과 상태 코드를 짝지어 봅니다.

In [ ]:
# 예제 1-3 (1) 오류 세 가지를 일부러 일으켜 봅니다
# 없는 모델과 빈 메시지는 기본 클라이언트로 보냅니다
tests = {
    '없는 모델': dict(model='claude-sonnet-9', max_tokens=50,
                      messages=[{'role': 'user', 'content': 'ok'}]),
    '빈 메시지': dict(model=SMALL_MODEL, max_tokens=50, messages=[]),
}
for name, request in tests.items():
    try:
        client.messages.create(**request)
    except anthropic.APIStatusError as error:      # 서버가 오류 코드로 응답한 경우
        print(f'{name} → {type(error).__name__} {error.status_code}')

# 틀린 키는 새 클라이언트로 보냅니다 · max_retries=0 은 SDK 자동 재시도를 끕니다
wrong_key_client = anthropic.Anthropic(api_key='sk-ant-wrong-key', max_retries=0)
try:
    wrong_key_client.messages.create(
        model=SMALL_MODEL, max_tokens=5, messages=[{'role': 'user', 'content': 'ok'}])
except anthropic.APIStatusError as error:
    print(f'틀린 키 → {type(error).__name__} {error.status_code}')

# Claude Code 에 물어보기 ·
#   「max_tokens 를 0 으로 보내면 어떤 오류인지 실행해 줘」

SDK 클라이언트에 기본으로 들어 있는 재시도 횟수와 시간 제한을 확인하고, 시간 제한을 0.01초로 줄여 시간 초과를 일으킵니다.
- **with_options**: 기존 클라이언트에서 일부 설정만 바꾼 복사본을 만듭니다. 원래 `client` 는 그대로 남습니다.
- **APIConnectionError**: 서버에 닿지 못했거나 제한 시간 안에 응답을 받지 못했을 때의 예외입니다. 시간 초과(`APITimeoutError`)도 여기에 포함됩니다.

출력에서 기본값과, 0.01초 제한일 때의 예외 이름을 확인합니다.

In [ ]:
# 예제 1-3 (2) SDK 에는 재시도와 시간 제한이 이미 들어 있습니다
print('기본 재시도 횟수', client.max_retries)
print('기본 시간 제한  ', client.timeout)

# 시간 제한을 0.01초로 줄인 복사본으로 보내 시간 초과를 일으킵니다
# 시간 초과(APITimeoutError)도 APIConnectionError 에 포함됩니다
short_timeout_client = client.with_options(timeout=0.01, max_retries=0)
try:
    short_timeout_client.messages.create(
        model=SMALL_MODEL, max_tokens=5, messages=[{'role': 'user', 'content': 'ok'}])
except anthropic.APIConnectionError as error:
    print('0.01초 제한 →', type(error).__name__)

# Claude Code 에 물어보기 ·
#   「timeout 을 0.5초씩 늘려 가며 통과하는 값을 찾아 줘」

한 번 호출이 실패할 확률을 6% 로 가정하고, 재시도 횟수에 따라 끝까지 실패로 남는 비율을 계산합니다.
- 실패가 서로 독립이면 attempts번 모두 실패할 확률은 `fail_rate ** attempts`(fail_rate 를 attempts번 곱한 값)입니다.

출력에서 세 번까지 보내면 120건 중 실패가 몇 건으로 줄어드는지 봅니다.

In [ ]:
# 예제 1-3 (3) 몇 번까지 다시 보낼지 계산합니다 · 한 번 실패 확률을 6% 로 가정
fail_rate = 0.06
# attempts 번 모두 실패할 확률은 fail_rate 를 attempts 번 곱한 값입니다
for attempts in (1, 2, 3, 4):
    remaining = fail_rate ** attempts          # 끝까지 실패로 남는 비율
    print(f'{attempts}번까지 보내면 남는 실패 {remaining * 100:6.3f}% · '
          f'120건 중 {remaining * 120:.2f}건')

# Claude Code 에 물어보기 ·
#   「fail_rate 20% 면 몇 번까지 보내야 1건 아래인지 계산해 줘」

### Claude Code 로 오류 처리 코드를 고칩니다 · 예제 1-3 (4) ~ (8)
(4) · (6) · (8) 은 PC 의 Claude Code 에서 하고, 노트북에서는 (5) · (7) 의 검사 셀만 실행합니다. (4) 는 코드 설명 듣기, (6) 은 기준을 주고 고치게 하기, (8) 은 검사가 보지 못하는 빈틈을 묻고 기준을 하나 더해 다시 고치게 하기입니다.
- **실습 폴더 `day02-cc`**: 환경 준비 (1) 에서 받은 재료 안에 있습니다. PC 에서는 따로 받은 `day02-cc` 폴더를 Claude Code 의 작업 폴더로 엽니다.
- **send_batch.py**: 문의를 한 건씩 보내 결과를 저장하는 코드입니다. 오류 처리가 하나도 없습니다. Claude Code 에 이 파일을 설명하고 고치게 합니다.
- **fake_api.py · check.py**: 가짜 응답기와 검사 스크립트입니다. 실제 서버에 보내지 않고 429 · 529 · 시간 초과 · 404 · 401 · 잘린 응답 · 거절을 흉내 냅니다. API 키가 없어도 되고 요금이 들지 않습니다.
- **send_batch_fixed.py**: (4) · (6) · (8) 을 Claude Code 로 진행해 고친 예시 파일입니다. 환경 준비 (1) 에서 함께 풀리며, (7) 에서 이 파일을 검사합니다.
- **prompts 폴더**: (4) · (6) · (8) 에서 Claude Code 에 보낼 글이 `예제1-3_1_설명.txt` · `예제1-3_2_수정.txt` · `예제1-3_3_검사가_못_보는_것.txt` · `예제1-3_4_기준_더하기.txt` 로 들어 있습니다. 파일 내용을 복사해 Claude Code 입력창에 붙여 넣습니다.

Claude Code 에 고치게 하기 전에 지금의 `send_batch.py` 를 검사합니다.
- `!` 로 시작하는 줄은 노트북 안에서 터미널 명령을 실행합니다. `python day02-cc/check.py` 는 검사 스크립트를 실행합니다.

출력에서 열 가지 검사 중 몇 개가 실패하는지, 어떤 항목이 실패하는지 봅니다.

In [ ]:
# 예제 1-3 (5) 고치기 전에 send_batch.py 를 검사합니다 · 가짜 응답기라 요금이 들지 않습니다
!python day02-cc/check.py

# Claude Code 에 물어보기 ·
#   「실패한 7개 검사가 send_batch.py 의 어느 줄 때문인지 짚어 줘」

PC 의 Claude Code 로 고친 것과 같은 일을 하는 예시 파일 `day02-cc/send_batch_fixed.py` 로 같은 검사를 다시 실행합니다. 이 파일은 환경 준비 (1) 에서 재료와 함께 풀려 있으니 따로 올리지 않습니다.
- 명령 끝의 파일 이름은 「이 파일을 검사하라」는 뜻입니다. 빼면 고치기 전 원본을 검사합니다.
- PC 의 내 `send_batch.py` 와 이 파일을 나란히 열어 `ask()` 가 어떻게 다른지 봅니다.

출력 마지막 줄이 `결과 10/10 통과` 인지 봅니다.

In [ ]:
# 예제 1-3 (7) Claude Code 로 고친 예시 파일을 다시 검사합니다 · 환경 준비 (1) 이 풀어 둔 send_batch_fixed.py
!python day02-cc/check.py day02-cc/send_batch_fixed.py

# Claude Code 에 물어보기 ·
#   「send_batch_fixed.py 에서 기다리는 시간을 계산하는 줄을 찾아 줘」

### 노트북으로 돌아와 같은 기준을 실제 API 에서 확인합니다 · 예제 1-3 (9) ~ (12)
Claude Code 가 고친 함수와 같은 기준으로 만든 `ask()` · `check_response()` 를 노트북에 두고, 실제 서버에 시간 초과 · 404 · 잘린 응답을 보내 봅니다. 이후 예제와 미션은 이 함수로 호출합니다.

예제 1-3 (6) 에서 Claude Code 에 준 기준과 같은 기준으로 `ask()` 를 만듭니다. Claude Code 가 고친 `send_batch.py` 의 `ask()` 와 나란히 놓고 비교해 봅니다.
- **다시 보낼 오류**: 요청 한도 초과(`RateLimitError`, 429), 서버 과부하(`OverloadedError`, 529), 그 밖의 서버 오류(`InternalServerError`, 5xx), 연결 오류와 시간 초과(`APIConnectionError`)입니다. 없는 모델(404)이나 틀린 키(401)는 다시 보내도 결과가 같아 바로 멈춥니다.
- `raise` 는 잡은 예외를 다시 일으켜 호출한 쪽으로 넘깁니다. `**extra` 는 나머지 인자를 그대로 `create` 에 넘깁니다.

함수를 정의만 하는 셀이라 출력이 없습니다.

In [ ]:
# 예제 1-3 (9) 다시 보낼 오류일 때만 재시도하는 ask() 를 만듭니다
# 529 는 OverloadedError · 시간 초과는 APIConnectionError 에 포함
RETRY = (anthropic.RateLimitError, anthropic.OverloadedError,
         anthropic.InternalServerError, anthropic.APIConnectionError)

def ask(system, user, model=SMALL_MODEL, max_tokens=300, tries=3, timeout=30, **extra):
    """요청을 보내 응답을 돌려줍니다 · 다시 보낼 오류면 tries 번까지 재시도합니다."""
    # SDK 재시도는 끄고 이 함수가 직접 다시 보냅니다
    no_retry_client = client.with_options(max_retries=0, timeout=timeout)
    request = dict(model=model, max_tokens=max_tokens,
                   messages=[{'role': 'user', 'content': user}], **extra)
    if system:
        request['system'] = system
    for i in range(tries):
        try:
            return no_retry_client.messages.create(**request)
        except RETRY as error:
            if i == tries - 1:
                raise                      # 마지막 시도면 예외를 그대로 넘깁니다
            wait_seconds = 2 ** i          # 1초, 2초 … 두 배씩 기다립니다
            print(f'  {type(error).__name__} → {wait_seconds}초 뒤 다시')
            time.sleep(wait_seconds)

# Claude Code 에 물어보기 ·
#   「tries 를 5 로 하면 기다림 총합이 몇 초인지 계산해 줘」

`ask()` 를 정상 요청, 시간 초과, 없는 모델로 각각 실행해 재시도 동작을 확인합니다.

출력에서 시간 초과는 `1초 뒤 다시`, `2초 뒤 다시` 를 거쳐 세 번 모두 실패하고, 없는 모델은 재시도 없이 바로 404 로 멈추는 것을 봅니다. 시간 초과의 마지막 예외 이름은 `APITimeoutError` 나 `APIConnectionError` 로 나오고, 둘 다 다시 보낼 오류로 처리됩니다.

In [ ]:
# 예제 1-3 (10) 정상 호출 한 번, 시간 초과는 다시 보내고, 없는 모델(404)은 바로 멈춥니다
# 정상 호출
response = ask(SYS_CLS, body)
print(text_of(response))

# 시간 초과 · 시간 초과(APITimeoutError)도 APIConnectionError 에 포함됩니다
try:
    ask(SYS_CLS, body, timeout=0.01)
except anthropic.APIConnectionError as error:
    print('세 번 다', type(error).__name__, '→ 실패 목록으로')

# 없는 모델
try:
    ask(SYS_CLS, body, model='claude-sonnet-9')
except anthropic.NotFoundError as error:
    print('없는 모델 → 다시 보내지 않고 바로 멈춤', error.status_code)

# Claude Code 에 물어보기 ·
#   「timeout=0.01 테스트가 총 몇 초 걸렸는지 재도록 고쳐 줘」

예외 없이 응답이 와도 쓸 수 없는 경우를 찾는 함수 `check_response()` 를 만듭니다.
- `stop_reason` 이 `max_tokens` 면 잘린 응답이고, `refusal` 이면 안전 사유로 거절된 응답입니다. 응답에 `{` 가 없으면 JSON 이 빠진 것으로 봅니다. ```json 으로 감싼 답은 이 검사를 통과합니다. 감싸기와 목록 밖 값은 예제 2-1 의 구조화 출력이 막습니다.
- 함수가 `None` 을 돌려주면 검사를 통과한 것입니다.

출력에서 `max_tokens=8` 로 보낸 응답이 예외 없이 `잘림` 으로 걸리는 것을 확인합니다.

In [ ]:
# 예제 1-3 (11) 예외가 나지 않는 실패를 검사합니다 · 잘림 · 거절 · JSON 없음
def check_response(response):
    """응답을 받아 쓸 수 없는 사유를 돌려줍니다 · 통과하면 None 을 돌려줍니다."""
    if response.stop_reason == 'max_tokens':
        return '잘림'
    if response.stop_reason == 'refusal':          # 안전 사유로 거절된 응답
        return '거절'
    if '{' not in text_of(response):
        return 'JSON 없음'
    return None

# max_tokens 를 8 로 줄여 잘린 응답을 받아 봅니다
response = ask(SYS_CLS, body, max_tokens=8)
print('예외 없음 · 검사 결과', check_response(response), '·', repr(text_of(response)))

# Claude Code 에 물어보기 ·
#   「```json 으로 감싼 답을 거르는 규칙을 하나 더해 줘」

지금까지 만든 함수를 한 흐름으로 묶어 20건을 처리합니다. 이 셀에서 만드는 `is_valid()` 로 거르고, `ask()` 로 보내고, `check_response()` 로 검사해 성공·실패 목록과 로그를 남깁니다.
- `inquiries['문의내용'].map(is_valid)` 는 열의 값마다 함수를 적용해 True/False 열을 만듭니다. 이 열로 표를 인덱싱하면 True 인 행만 남습니다.
- **anthropic.APIError**: API 호출에서 나는 예외의 부모 클래스입니다. 재시도 끝에 실패한 호출은 모두 여기서 잡혀 실패 목록으로 갑니다.

출력에서 건너뜀·성공·실패 건수와 로그 첫 줄(id, 모델, 입력 토큰, 출력 토큰, 종료 이유)을 봅니다.

In [ ]:
# 예제 1-3 (12) 거르고, 보내고, 실패를 따로 모읍니다 · 20건
def is_valid(text):
    """문의 글을 받아 5자 이상의 문자열인지 돌려줍니다."""
    return isinstance(text, str) and len(text.strip()) >= 5

# 보낼 만한 문의만 남깁니다
work = inquiries[inquiries['문의내용'].map(is_valid)]

# 20건을 보내고 성공 · 실패 · 로그를 따로 모읍니다
success, failed, log = [], [], []
for _, row in work.head(20).iterrows():
    prompt = f"id: {row['id']}\n문의: {row['문의내용']}"
    try:
        response = ask(SYS_CLS, prompt)
        why = check_response(response)
        if why:                                # 검사에 걸리면 실패 목록으로
            failed.append(row['id'])
        else:
            success.append(row['id'])
        usage = response.usage
        log.append((row['id'], response.model, usage.input_tokens,
                    usage.output_tokens, response.stop_reason))
    except anthropic.APIError:                 # 재시도 끝에 실패한 호출
        failed.append(row['id'])
print('건너뜀', len(inquiries) - len(work), '· 성공', len(success), '· 실패', len(failed))
print('로그 첫 줄', log[0])

# Claude Code 에 물어보기 ·
#   「log 를 DataFrame 으로 만들어 출력 토큰 합계를 내 줘」

## 예제 2-1 · 텍스트에서 구조화 데이터 추출
같은 문의를 형식 지정 없이, 프롬프트로, 구조화 출력(JSON 스키마)으로 받아 보고 파싱이 어디서 깨지는지 비교합니다.

형식을 정하지 않고 분류를 요청해 응답이 어떤 모양으로 오는지 봅니다. 여기서부터는 예제 1-3 (9) 의 `ask()` 로 호출합니다. 첫 인자 `None` 은 system 없이 보낸다는 뜻입니다. `body`(Q101 문의)와 `SYS_CLS`(분류 규칙)는 예제 1-1 (4) 에서 만들었습니다. `NameError` 가 나면 그 셀부터 다시 실행합니다.

출력에서 분류 결과가 문장 속에 섞여 있어 코드로 값을 꺼내기 어렵다는 점을 확인합니다.

In [ ]:
# 예제 2-1 (1) 형식을 정하지 않으면 문장으로 답합니다
response = ask(None, '다음 문의를 분류해줘.\n' + body)     # system 없이 보냅니다
print(text_of(response))

# Claude Code 에 물어보기 ·
#   「이 마크다운 답에서 우선순위 값을 꺼내는 코드를 써 줘」

system 에 JSON 형식을 지시해 받은 응답을 직접 파싱합니다.
- **파싱**: 글자로 된 데이터를 프로그램이 다룰 수 있는 값으로 바꾸는 일입니다. `json.loads` 는 JSON 글자를 딕셔너리로 바꿉니다.
- 응답 앞뒤에 설명이나 코드 블록 표시가 붙을 수 있어 첫 `{` 부터 마지막 `}` 까지만 잘라 읽습니다.

출력에서 `repr` 로 찍은 원문에 JSON 외의 글자가 붙었는지, `urgent` 가 `bool` 타입으로 읽혔는지 봅니다.

In [ ]:
# 예제 2-1 (2) 프롬프트로 JSON 을 요청하고 직접 파싱합니다
response = ask(SYS_CLS, body)
answer_text = text_of(response)
print(repr(answer_text))

# 앞뒤에 무엇이 붙어도 첫 { 부터 마지막 } 까지만 잘라 읽습니다
start = answer_text.find('{')
end = answer_text.rfind('}') + 1
result = json.loads(answer_text[start:end])
print(result['category'], result['urgent'], type(result['urgent']))

# Claude Code 에 물어보기 ·
#   「앞에 설명 문장이 붙은 답으로 파싱 코드를 테스트해 줘」

응답 형식을 JSON 스키마로 정해 그 모양의 JSON 만 받는 구조화 출력을 씁니다.
- **JSON 스키마**: JSON 의 모양을 적는 규격입니다. `properties` 에 키와 타입을, `required` 에 반드시 있어야 할 키를 적습니다.
- **enum**: 값이 될 수 있는 목록입니다. 목록 밖의 값은 나오지 않습니다.
- **output_config**: `{'format': {'type': 'json_schema', 'schema': ...}}` 를 넣으면 구조화 출력이 켜집니다. `ask()` 의 `**extra` 를 거쳐 `create` 로 전달됩니다.

출력에서 원문이 앞뒤 글자 없이 JSON 그대로이고, 자르지 않아도 `json.loads` 가 되는 것을 확인합니다.

In [ ]:
# 예제 2-1 (3) 구조화 출력 · 스키마를 주면 그 모양의 JSON 만 돌아옵니다
# enum 은 허용값 목록, required 는 반드시 있어야 할 키입니다
SCHEMA = {
    'type': 'object',
    'properties': {
        'id':       {'type': 'string'},
        'category': {'type': 'string',
                     'enum': ['문서안내', '절차안내', '긴급', '데이터요청']},
        'urgent':   {'type': 'boolean'},
    },
    'required': ['id', 'category', 'urgent'],
    'additionalProperties': False,     # 목록에 없는 키는 받지 않습니다
}
FMT = {'format': {'type': 'json_schema', 'schema': SCHEMA}}

# output_config 로 스키마를 넘겨 보냅니다
response = ask(SYS_CLS, body, output_config=FMT)
answer_text = text_of(response)
print(repr(answer_text))
print(json.loads(answer_text))             # 자르지 않고 그대로 읽습니다

# Claude Code 에 물어보기 ·
#   「enum 밖 값을 유도하는 문의를 보내 결과를 보여 줘」

장비명, 현상, 조치 구분을 문장 하나에서 한 번에 뽑는 스키마를 씁니다.
- `'type': ['string', 'null']` 은 문자열이나 null(값 없음)을 허용한다는 뜻입니다. null 을 허용해 두고 system 에서 없는 값은 null 로 두라고 지시하면, 문장에 장비명이 없을 때 임의의 값 대신 null 을 받습니다.
- 파이썬으로 읽으면 null 은 `None` 이 됩니다.

출력에서 두 번째 문의의 `equipment` 가 `None` 으로 나오는지 봅니다.

In [ ]:
# 예제 2-1 (4) 문장에서 여러 값을 한 번에 뽑습니다 · 없으면 null
EXTRACT = {
    'type': 'object',
    'properties': {
        'equipment': {'type': ['string', 'null']},     # 장비명 · 없으면 null
        'symptom':   {'type': 'string'},               # 현상 한 줄
        'action':    {'type': 'string',
                      'enum': ['즉시조치', '확인요청', '자료요청', '안내']},
    },
    'required': ['equipment', 'symptom', 'action'],
    'additionalProperties': False,
}
extract_system = '현장 문의에서 값을 뽑는다. 문의에 없는 값은 null 로 둔다.'
extract_format = {'format': {'type': 'json_schema', 'schema': EXTRACT}}

# 문의 두 건을 보내 뽑힌 값을 봅니다
inquiry_texts = ['5호기 챔버 압력 알람이 지속됩니다',
                 'PM 체크리스트 최신본 위치를 알고 싶습니다']
for inquiry_text in inquiry_texts:
    response = ask(extract_system, inquiry_text, output_config=extract_format)
    print(json.loads(text_of(response)))

# Claude Code 에 물어보기 ·
#   「symptom 도 null 허용으로 바꾸면 결과가 어떻게 되는지 돌려 줘」

같은 스키마를 딕셔너리 대신 파이썬 클래스로 적고, 응답을 그 클래스의 객체로 받습니다.
- **pydantic `BaseModel`**: pydantic 라이브러리의 클래스입니다. 필드 이름과 타입을 적으면 값 검사와 변환을 해 줍니다. 이 클래스에서 JSON 스키마가 만들어집니다.
- **Literal**: 허용값을 나열하는 타입입니다. JSON 스키마의 `enum` 과 같은 역할을 합니다.
- **messages.parse**: `output_format` 에 클래스를 넘기면 응답을 파싱해 `parsed_output` 에 객체로 담아 돌려줍니다. 예제 2-1 (3) 의 `output_config` 와 인자 이름이 다릅니다.

출력에서 `response.parsed_output.category` 처럼 점으로 값을 읽는 것을 확인합니다.

In [ ]:
# 예제 2-1 (5) 파이썬 클래스로 스키마를 적는 방법 · messages.parse
from pydantic import BaseModel
from typing import Literal

class Inquiry(BaseModel):          # 필드 이름과 타입이 스키마가 됩니다
    id: str
    category: Literal['문서안내', '절차안내', '긴급', '데이터요청']
    urgent: bool

# output_format 에 클래스를 넘기면 응답이 parsed_output 에 객체로 담겨 옵니다
response = client.messages.parse(
    model=SMALL_MODEL, max_tokens=300, system=SYS_CLS,
    messages=[{'role': 'user', 'content': body}], output_format=Inquiry)
print(response.parsed_output)             # 검사를 통과한 Inquiry 객체
print(response.parsed_output.category, response.parsed_output.urgent)

# Claude Code 에 물어보기 ·
#   「Inquiry 클래스에서 만들어지는 JSON 스키마를 찍어 줘」

## 예제 2-2 · 엑셀·DB 대량 데이터 일괄 처리
120건 표를 읽어 구조화 출력으로 한 번에 분류하고, 결과를 엑셀 두 시트와 SQLite DB 에 저장합니다. 두 번 돌려도 행이 늘지 않게 만듭니다.

120건 표를 다시 읽고 `is_valid()` 로 보낼 문의만 남깁니다. 예제 1-3 (12) 와 같은 거르기입니다.

출력에서 처리 대상과 건너뛴 건수를 확인합니다.

In [ ]:
# 예제 2-2 (1) 표를 읽고 보낼 것만 남깁니다
inquiries = pd.read_csv('inquiries_120.csv').fillna('')
work = inquiries[inquiries['문의내용'].map(is_valid)]
skipped = len(inquiries) - len(work)
print('전체', len(inquiries), '· 처리 대상', len(work), '· 건너뜀', skipped)

# Claude Code 에 물어보기 ·
#   「건너뛴 2건의 id 와 원문을 보여 줘」

문의 한 건을 구조화 출력으로 분류하는 `classify()` 와, 실패해도 멈추지 않게 감싼 `safe()` 를 만듭니다.
- `result.update(...)` 는 딕셔너리에 키를 더합니다. 토큰 수와 종료 이유를 함께 남겨 요금 계산과 검증에 씁니다.
- `except Exception` 은 거의 모든 예외를 잡습니다. 실패한 건은 `error` 키에 예외 이름을 담아 결과로 돌려줍니다.

출력에서 첫 건의 분류 결과 딕셔너리를 확인합니다.

In [ ]:
# 예제 2-2 (2) 한 건을 분류하는 함수 · 구조화 출력 + 사용량 기록
def classify(x, model=SMALL_MODEL):
    """문의 한 행을 구조화 출력으로 분류해 결과 딕셔너리를 돌려줍니다."""
    user = f"id: {x['id']}\n문의: {x['문의내용']}"
    response = ask(SYS_CLS, user, model=model, output_config=FMT)
    result = json.loads(text_of(response))
    # 요금 계산과 검증에 쓸 토큰 수와 종료 이유를 붙입니다
    result.update(in_tok=response.usage.input_tokens,
                  out_tok=response.usage.output_tokens,
                  stop=response.stop_reason)
    return result

def safe(x):
    """classify() 가 실패하면 오류 이름을 담은 딕셔너리를 돌려줍니다."""
    try:
        return classify(x)
    except Exception as error:             # 한 건이 실패해도 배치는 계속 갑니다
        return {'id': x['id'], 'error': type(error).__name__}

print(safe(work.iloc[0]))

# Claude Code 에 물어보기 ·
#   「safe 가 error 를 돌려주는 상황을 하나 만들어 보여 줘」

처리 대상 118건을 `ThreadPoolExecutor` 로 여덟 건씩 동시에 보냅니다(예제 1-2 (6)).
- `pd.DataFrame(rows)` 는 딕셔너리 목록을 표로 바꿉니다. 딕셔너리의 키가 열 이름이 됩니다.
- `value_counts()` 는 열의 값마다 몇 건인지 셉니다.

출력에서 걸린 시간, 오류 건수, 분류별 건수를 봅니다.

In [ ]:
# 예제 2-2 (3) 118건을 여덟 개씩 동시에 보냅니다
started = time.time()
work_rows = [row for _, row in work.iterrows()]
with ThreadPoolExecutor(max_workers=8) as executor:
    rows = list(executor.map(safe, work_rows))
first_run = pd.DataFrame(rows)
seconds = time.time() - started
# error 열이 없으면 빈 Series 로 세어 0 이 나옵니다
error_count = first_run.get('error', pd.Series()).notna().sum()
print(len(first_run), '건 ·', f'{seconds:.1f}초 · 오류', error_count)
print(first_run['category'].value_counts().to_dict())

# Claude Code 에 물어보기 ·
#   「rows 의 out_tok 합계로 118건 실제 요금을 계산해 줘」

분류 결과를 원본 표에 id 로 붙이고, 엑셀 파일 하나에 전체 시트와 확인필요 시트를 저장합니다.
- **merge**: 두 표를 같은 열(`on='id'`)의 값으로 이어 붙입니다. `how='left'` 는 원본 표의 행을 모두 남기고, 결과가 없는 행은 빈칸(NaN)으로 둡니다.
- **ExcelWriter**: 엑셀 파일 하나를 열어 두고 `to_excel(..., sheet_name=...)` 으로 시트를 여러 장 씁니다. `with` 블록이 끝나면 파일이 저장됩니다.

출력에서 시트 이름별 행 수를 확인합니다.

In [ ]:
# 예제 2-2 (4) 원본 표에 붙여 엑셀 두 시트로 저장합니다
RUN_AT = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')
# 결과 없는 행은 빈칸으로 남습니다
merged = inquiries.merge(first_run, on='id', how='left')
merged['model'] = SMALL_MODEL
merged['run_at'] = RUN_AT
# 전체 시트와 확인필요 시트를 파일 하나에 씁니다
with pd.ExcelWriter('문의_분류결과.xlsx') as writer:
    merged.to_excel(writer, sheet_name='전체', index=False)
    need_review = merged[merged['category'].isna()]
    need_review.to_excel(writer, sheet_name='확인필요', index=False)
sheets = pd.read_excel('문의_분류결과.xlsx', sheet_name=None)
print({key: len(value) for key, value in sheets.items()})

# Claude Code 에 물어보기 ·
#   「확인필요 시트에 들어간 2건이 왜 빈칸인지 확인해 줘」

분류 결과를 SQLite DB 파일 `inquiries.db` 에 저장합니다.
- **SQLite**: 파일 하나로 동작하는 데이터베이스입니다. 파이썬에 `sqlite3` 모듈로 들어 있어 따로 설치하지 않습니다.
- **PRIMARY KEY(기본 키)**: 행을 구별하는 열입니다. 같은 id 를 두 번 넣을 수 없습니다. `INSERT OR REPLACE` 는 같은 id 가 있으면 새 값으로 덮어씁니다.
- `?` 자리에는 `executemany` 에 넘긴 튜플의 값이 차례로 들어갑니다. `commit()` 을 해야 파일에 반영됩니다.

출력에서 분류별 행 수를 확인합니다.

In [ ]:
# 예제 2-2 (5) SQLite DB 에 저장합니다 · id 를 기본 키로 두고 INSERT OR REPLACE
import sqlite3
db = sqlite3.connect('inquiries.db')
db.execute('''CREATE TABLE IF NOT EXISTS classified (
    id TEXT PRIMARY KEY, category TEXT, urgent INTEGER, model TEXT, run_at TEXT)''')
success_rows = [row for row in rows if 'error' not in row]
# ? 자리에 값이 차례로 들어가고, 같은 id 는 덮어씁니다
INSERT_SQL = 'INSERT OR REPLACE INTO classified VALUES (?, ?, ?, ?, ?)'
db_rows = [(row['id'], row['category'], int(row['urgent']), SMALL_MODEL, RUN_AT)
           for row in success_rows]
db.executemany(INSERT_SQL, db_rows)
db.commit()                 # 커밋해야 파일에 반영됩니다
by_category = db.execute(
    'SELECT category, COUNT(*) FROM classified GROUP BY category').fetchall()
print(by_category)

# Claude Code 에 물어보기 ·
#   「urgent 가 1 인 행만 SQL 로 조회해 줘」

같은 결과를 한 번 더 넣어 행 수가 그대로인지 보고, DB 에 이미 있는 id 를 빼 다음 실행에서 처리할 문의만 고릅니다.
- `{inquiry_id for (inquiry_id,) in ...}` 는 조회 결과에서 id 만 꺼내 집합(set)을 만듭니다.
- `isin(done_ids)` 은 값이 집합 안에 있는지 검사하고, 앞의 `~` 는 True 와 False 를 뒤집습니다.

출력에서 두 번 넣은 뒤에도 행 수가 같은지, 다음 실행에서 처리할 것이 몇 건인지 확인합니다.

In [ ]:
# 예제 2-2 (6) 두 번 넣어도 행 수가 그대로인지, 남은 건만 골라낼 수 있는지 봅니다
db.executemany(INSERT_SQL, db_rows)       # 같은 결과를 한 번 더 넣습니다
db.commit()
row_count = db.execute('SELECT COUNT(*) FROM classified').fetchone()[0]
print('두 번 넣은 뒤 행 수', row_count)
# DB 에 있는 id 집합을 만들고, 거기에 없는 행만 남깁니다
id_rows = db.execute('SELECT id FROM classified')
done_ids = {inquiry_id for (inquiry_id,) in id_rows}
remaining = work[~work['id'].isin(done_ids)]     # ~ 는 뒤집기 · done_ids 에 없는 행
print('이미 처리', len(done_ids), '· 다음 실행에서 처리할 것', len(remaining))

# Claude Code 에 물어보기 ·
#   「INSERT OR REPLACE 를 INSERT 로 바꾸면 무슨 오류인지 보여 줘」

## 예제 2-3 · 출력 검증과 실패 건 처리
형식이 맞아도 틀릴 수 있습니다. 검증 규칙을 함수 하나로 모으고, 다시 돌려 답이 갈린 건을 찾아 사람이 확인할 목록을 만듭니다.

형식은 맞지만 쓸 수 없는 응답을 걸러 내는 규칙을 `validate()` 하나에 모읍니다. 걸리면 사유 문자열을, 통과하면 `None` 을 돌려줍니다.
- `o.get('키')` 는 키가 없을 때 오류 대신 `None` 을 돌려줍니다.
- 마지막 규칙은 업무 규칙입니다. `urgent` 가 true 인데 분류가 `긴급` 이 아니면 서로 맞지 않는 응답으로 봅니다.

함수를 정의만 하는 셀이라 출력이 없습니다.

In [ ]:
# 예제 2-3 (1) 검증 규칙을 함수 하나로 모읍니다
CATS = {'문서안내', '절차안내', '긴급', '데이터요청'}

def validate(o, want_id):
    """분류 결과 하나를 검사해 걸린 사유를, 통과하면 None 을 돌려줍니다."""
    if 'error' in o:
        return '호출 실패'
    if o.get('stop') != 'end_turn':
        return '응답 잘림'
    if o.get('id') != want_id:
        return 'id 불일치'
    if o.get('category') not in CATS:
        return '분류값 이상'
    if o['urgent'] and o['category'] != '긴급':
        return '긴급 표시 불일치'     # 형식은 맞지만 업무 규칙에 어긋남
    return None

# Claude Code 에 물어보기 ·
#   「긴급인데 urgent 가 false 인 경우도 거르는 규칙을 더해 줘」

118건의 결과를 `validate()` 로 검사해 사유별 건수를 셉니다.
- **Counter**: 목록의 값마다 몇 번 나왔는지 세는 파이썬 표준 도구입니다.
- `dropna()` 는 빈 값(통과한 건의 `None`)을 빼고, `isna().sum()` 은 빈 값의 수, 곧 통과 건수를 셉니다.

출력에서 걸린 사유와 통과 건수를 확인합니다.

In [ ]:
# 예제 2-3 (2) 사유별로 셉니다
from collections import Counter
reasons = []
for result, inquiry_id in zip(rows, work['id']):
    reasons.append(validate(result, inquiry_id))   # 통과하면 None
first_run['why'] = reasons
print(Counter(first_run['why'].dropna()))
print('통과', first_run['why'].isna().sum(), '/', len(first_run))

# Claude Code 에 물어보기 ·
#   「긴급 표시 불일치로 걸린 건의 원문과 결과를 보여 줘」

같은 118건을 한 번 더 보내고, 두 결과의 분류가 다른 건을 찾습니다. 같은 입력에도 답이 갈리는 건은 분류 기준이 애매한 문의일 가능성이 큽니다.
- `.values` 로 두 열을 배열로 꺼내 같은 순서끼리 비교합니다. 비교 결과(True/False)로 표를 인덱싱하면 다른 행만 남습니다.

출력에서 갈린 건수와 해당 문의의 앞부분을 봅니다.

In [ ]:
# 예제 2-3 (3) 같은 118건을 한 번 더 돌려 답이 갈리는 건을 찾습니다
with ThreadPoolExecutor(max_workers=8) as executor:
    second_rows = list(executor.map(safe, work_rows))
second_run = pd.DataFrame(second_rows)
# 같은 순서끼리 비교해 분류가 다른 행만 남깁니다
changed = first_run['category'].values != second_run['category'].values
disagreed = first_run[changed]
print('두 번 돌려 분류가 갈린 건', len(disagreed), '/', len(first_run))
work_by_id = work.set_index('id')
for inquiry_id in disagreed['id']:
    print(' ', inquiry_id, work_by_id.loc[inquiry_id, '문의내용'][:40])

# Claude Code 에 물어보기 ·
#   「두 번 돌린 결과에서 urgent 가 갈린 건도 세어 줘」

답이 갈렸거나 검증에 걸린 건만 모아 중형 모델로 다시 분류합니다. 전체를 중형 모델로 돌릴 때보다 요금이 적게 듭니다.
- `set(a) | set(b)` 는 두 집합의 합집합입니다. 양쪽에 다 있는 id 는 한 번만 들어갑니다.
- 딕셔너리끼리의 `|` 는 두 딕셔너리를 합칩니다. 여기서는 행 데이터에 `id` 키를 붙입니다.

중형 모델의 답은 표에 적지 않습니다. 이 건들은 예제 2-3 (5) 에서 모두 `review` 로 표시하고, 사람이 확인할 때 참고합니다.

출력에서 경량 모델 1차·2차 분류와 중형 모델 분류를 나란히 비교합니다.

In [ ]:
# 예제 2-3 (4) 갈렸거나 검증에 걸린 건만 중형 모델로 다시 봅니다
invalid_ids = set(first_run.loc[first_run['why'].notna(), 'id'])   # 검증에 걸린 id
recheck = set(disagreed['id']) | invalid_ids                        # | 는 합집합
# 골라낸 건만 중형 모델로 다시 분류합니다
medium_results = {}
for inquiry_id in recheck:
    # 행에 id 키를 붙여 classify() 에 넘깁니다
    row = work_by_id.loc[inquiry_id].to_dict() | {'id': inquiry_id}
    medium_results[inquiry_id] = classify(row, model=MODEL)
# 경량 1차 · 2차 · 중형 분류를 나란히 봅니다
first_by_id = first_run.set_index('id')
second_by_id = second_run.set_index('id')
for inquiry_id, result in medium_results.items():
    first = first_by_id.loc[inquiry_id, 'category']
    second = second_by_id.loc[inquiry_id, 'category']
    medium = result['category']
    print(f'{inquiry_id} 경량 1차 {first} · 2차 {second} · 중형 {medium}')

# Claude Code 에 물어보기 ·
#   「Q103 원문을 보고 세 분류 중 어느 것이 맞는지 근거를 말해 줘」

모든 행에 처리 상태를 붙입니다. `auto` 는 그대로 쓰는 건, `review` 는 사람이 확인할 건, `manual` 은 호출이 실패했거나 빈 문의라 보내지 않아 사람이 직접 처리할 건입니다.
- `dict(zip(first_run['id'], first_run['why']))` 는 id 를 키, 검증 사유를 값으로 하는 조회표(딕셔너리)를 만듭니다.
- `merged['id'].map(why_by_id)` 는 원본 120행의 id 마다 조회표에서 사유를 찾아 `why` 열로 붙입니다. 보내지 않은 행은 빈 값이 됩니다.
- 마지막 줄은 행마다 id 와 사유를 `status()` 에 넣어 `status` 열을 채웁니다.

출력에서 상태별 건수를 확인합니다. 엑셀의 확인필요 시트는 `auto` 가 아닌 행으로 다시 저장됩니다.

In [ ]:
# 예제 2-3 (5) 상태를 셋으로 나눕니다 · 자동 · 사람 확인 · 실패
def status(i, why):
    """id 와 검증 사유를 받아 처리 상태(auto · review · manual)를 돌려줍니다."""
    if why == '호출 실패':
        return 'manual'
    if i in recheck:
        return 'review'
    return 'auto'

# 행마다 id 와 검증 사유를 status() 에 넣습니다
why_by_id = dict(zip(first_run['id'], first_run['why']))   # id → 검증 사유 조회표
merged['why'] = merged['id'].map(why_by_id)                # 보내지 않은 행은 빈 값
statuses = []
for inquiry_id, why in zip(merged['id'], merged['why']):
    statuses.append(status(inquiry_id, why))
merged['status'] = statuses
merged.loc[merged['category'].isna(), 'status'] = 'manual'
print(merged['status'].value_counts().to_dict())
# 확인필요 시트는 auto 가 아닌 행으로 다시 저장합니다
with pd.ExcelWriter('문의_분류결과.xlsx') as writer:
    merged.to_excel(writer, sheet_name='전체', index=False)
    need_review = merged[merged['status'] != 'auto']
    need_review.to_excel(writer, sheet_name='확인필요', index=False)

# Claude Code 에 물어보기 ·
#   「review 4건의 id 와 why 열을 표로 보여 줘」

이번 실행의 시각, 모델, 프롬프트 앞부분, 스키마 키, 건수, 상태별 건수를 `run_meta.json` 에 저장합니다. 나중에 결과 숫자의 출처를 설명하거나 같은 조건으로 다시 실행할 때 이 기록을 봅니다.
- `json.dump(..., ensure_ascii=False)` 는 한글을 읽을 수 있는 글자 그대로 저장합니다.

출력에서 저장한 항목을 확인합니다.

In [ ]:
# 예제 2-3 (6) 실행 조건을 함께 남깁니다 · 나중에 「이 숫자 어떻게 나왔나」에 답하는 기록
meta = {'run_at': RUN_AT,
        'model': SMALL_MODEL,
        'prompt': SYS_CLS[:40] + '…',
        'schema_keys': list(SCHEMA['properties']),
        'n_input': len(inquiries),
        'n_called': len(work),
        'status': merged['status'].value_counts().to_dict()}
# 실행 조건을 run_meta.json 파일로 저장합니다
with open('run_meta.json', 'w', encoding='utf-8') as file:
    json.dump(meta, file, ensure_ascii=False, indent=1)
for key, value in meta.items():
    print(f'{key:11s}', value)

# Claude Code 에 물어보기 ·
#   「run_meta.json 을 읽어 지난 실행과 status 를 비교하는 코드를 써 줘」

## 예제 3-1 · 모델이 학습하지 않은 것을 직접 확인하기
가상 팹 SL-FAB 의 사내 규정을 근거 없이 물어보고, 같은 질문에 문서를 붙여 다시 물어 결과를 비교합니다.

가상 회사의 사내 규정 수치를 문서 없이 세 번 물어봅니다. 모델은 이 회사의 문서를 학습한 적이 없습니다.
- **ask_text()**: 질문 하나를 예제 1-3 의 `ask()` 로 보내 (답 글자, 사용량) 두 값을 돌려받습니다. 답만 쓸 때는 `ask_text(...)[0]` 입니다. 파트 3 에서 계속 씁니다.
- 시간 제한은 600초입니다. 문서를 많이 붙이는 예제 3-1 (4) 에서 씁니다. `{'effort': 'low'}` 는 짧은 답을 빠르게 받는 설정입니다(예제 1-1 (8)).
- `ask_text(Q0)[0]` 은 돌려받은 (답, 사용량) 중 답입니다. `[:90]` 으로 앞 90자만 자르고 줄바꿈을 공백으로 바꿔 한 줄로 출력합니다.

출력에서 세 답의 숫자가 서로 다른지, 모른다고 답하는지 봅니다.

In [ ]:
# 예제 3-1 (1) 사내 규정을 근거 없이 물어봅니다 · 세 번
# 짧은 답에는 생각을 줄여 빠르게 받습니다 · 예제 1-1 (8)
LOW = {'effort': 'low'}

def ask_text(prompt, max_tokens=600):
    """예제 1-3 의 ask() 로 질문을 보내 (답 글자, 사용량)을 돌려줍니다."""
    # 문서를 많이 붙이면 오래 걸려 600초까지 기다립니다
    response = ask(None, prompt, model=MODEL, max_tokens=max_tokens,
                   timeout=600, output_config=LOW)
    return text_of(response), response.usage

Q0 = 'SL-FAB 에치 장비의 ESC 온도 정상 운전 범위가 몇 도인가요? 숫자로 짧게 답해줘.'
for i in range(3):
    answer = ask_text(Q0)[0]                      # (답, 사용량) 중 답만 씁니다
    print(i + 1, answer[:90].replace('\n', ' '))

# Claude Code 에 물어보기 ·
#   「LOW 를 빼고 세 번 돌려 답과 출력 토큰 차이를 알려 줘」

같은 질문에 해당 SOP 문서 원문을 붙여 다시 묻습니다. 근거 문서를 프롬프트에 넣으면 모델은 그 내용을 바탕으로 답을 생성합니다.
- `open(..., encoding='utf-8').read()` 는 텍스트 파일 전체를 문자열 하나로 읽습니다.

출력에서 근거 없이 물었던 예제 3-1 (1) 과 달리 답에 기준 숫자가 들어 있는지, 입력 토큰이 얼마인지 봅니다.

In [ ]:
# 예제 3-1 (2) 같은 질문에 해당 문서를 붙여 물어봅니다
doc_text = open('corpus/SOP-ETCH-007_v2.1.md', encoding='utf-8').read()
prompt = f'다음 문서를 근거로 답해줘.\n\n{doc_text}\n\n질문: {Q0}'
answer, usage = ask_text(prompt)
print(answer)
print('입력 토큰', usage.input_tokens)

# Claude Code 에 물어보기 ·
#   「문서에 없는 질문을 붙여 보내면 뭐라고 답하는지 보여 줘」

문서 253건을 모두 붙였을 때의 입력 토큰을 보내기 전에 셉니다. 예제 1-2 (1) 의 `count_tokens` 입니다.
- `join` 은 문서 목록을 빈 줄을 사이에 두고 문자열 하나로 잇습니다.
- 요금은 중형 모델 입력 정가인 100만 토큰당 2달러로 계산합니다.

출력에서 질문 한 번의 입력 토큰과, 하루 1,000번 물을 때의 요금을 봅니다.

In [ ]:
# 예제 3-1 (3) 문서 253건을 전부 붙이면 몇 토큰일까요
docs_all = []
for file_name in sorted(os.listdir('corpus')):
    docs_all.append(open(f'corpus/{file_name}', encoding='utf-8').read())
all_docs_text = '\n\n'.join(docs_all)            # 문서 사이에 빈 줄을 둡니다
full_prompt = all_docs_text + '\n\n질문: ' + Q0
# 보내기 전에 입력 토큰만 셉니다
count = client.messages.count_tokens(
    model=MODEL, messages=[{'role': 'user', 'content': full_prompt}])
full_input_tokens = count.input_tokens
total_chars = sum(map(len, docs_all))
print('문서', len(docs_all), '건 · 글자', total_chars, '· 입력 토큰', full_input_tokens)
# 중형 모델 입력 정가 100만 토큰당 2달러로 계산합니다
cost_usd = full_input_tokens * 2 / 1e6
print(f'질문 한 번 입력 요금 ${cost_usd:.3f} · '
      f'하루 질문 1,000번이면 ${cost_usd * 1000:.0f}')

# Claude Code 에 물어보기 ·
#   「문서 종류별 토큰 합계를 count_tokens 로 세 줘」

253건을 실제로 모두 붙여 보내고 걸린 시간과 답을 봅니다.

출력의 입력 토큰을 예제 3-1 (2) 에서 문서 한 건만 붙였을 때(660 토큰 안팎)와 비교합니다. 문서를 골라 붙이는 검색을 왜 쓰는지 이 차이로 확인합니다.

In [ ]:
# 예제 3-1 (4) 실제로 전부 붙여 보내 봅니다 · 시간과 답을 봅니다
started = time.time()
prompt = '다음 문서들을 근거로 답해줘.\n\n' + all_docs_text + '\n\n질문: ' + Q0
answer, usage = ask_text(prompt)
seconds = time.time() - started
print(f'{seconds:.1f}초 · 입력 토큰 {usage.input_tokens}')
print(answer[:200])

# Claude Code 에 물어보기 ·
#   「2초면 빠른데 왜 토큰 요금이 문제인지 숫자로 설명해 줘」

## 예제 3-2 · 키워드 검색으로 먼저 찾아보기
현장에서 실제로 하는 질문을 낱말이 겹치는지로 찾는 키워드 검색(TF-IDF)에 넣어 봅니다.

corpus 폴더의 `.md` 문서를 모두 읽어 본문 목록 `docs` 와 이름 목록 `names` 를 만듭니다. 두 목록은 순서가 같아 같은 번호가 같은 문서입니다.

출력에서 문서 수와 전체 글자 수를 확인합니다.

In [ ]:
# 예제 3-2 (1) 코퍼스를 엽니다
# .md 파일만 이름순으로 고릅니다
file_names = []
for file_name in sorted(os.listdir('corpus')):
    if file_name.endswith('.md'):
        file_names.append(file_name)
# 본문 목록 docs 와 이름 목록 names 는 같은 순서입니다
docs = []
for file_name in file_names:
    docs.append(open(f'corpus/{file_name}', encoding='utf-8').read())
names = [file_name[:-3] for file_name in file_names]   # 확장자 .md 를 뗀 이름
total_chars = sum(len(doc_text) for doc_text in docs)
print(len(docs), '건 · 글자', total_chars)

# Claude Code 에 물어보기 ·
#   「문서 길이 분포(최소·중앙·최대 글자)를 계산해 줘」

문서 이름의 앞부분(SOP·MAN·SPEC 등)으로 종류별 건수를 세고, 문서 한 건의 앞부분을 봅니다.
- `collections.Counter` 는 값마다 나온 횟수를 세는 도구입니다. `name.split('-')[0]` 은 이름을 `-` 로 나눈 첫 조각입니다.

출력에서 문서 종류 분포와 SOP 문서의 모양을 확인합니다.

In [ ]:
# 예제 3-2 (2) 어떤 문서들인지 종류별로 셉니다
import collections
kinds = [name.split('-')[0] for name in names]     # 이름을 - 로 나눈 첫 조각
print(collections.Counter(kinds))
print(docs[names.index('SOP-ETCH-007_v2.1')][:120])

# Claude Code 에 물어보기 ·
#   「SOP 문서의 제목 줄만 모아 목록으로 보여 줘」

현장 작업자가 실제로 쓸 법한 말투의 질문 `Q` 와 그 정답 문서 이름 `GOLD` 를 정합니다. 질문은 일상 표현으로 적혀 있어 문서에 쓰인 낱말과 다를 수 있습니다.

출력이 없는 셀입니다. 다음 셀부터 이 질문으로 검색합니다.

In [ ]:
# 예제 3-2 (3) 현장에서 실제로 하는 질문과 그 정답 문서
Q = '칠러가 안 도는 것 같은데 챔버가 멈추는 기준이 뭔가요'
GOLD = 'SOP-ETCH-007_v2.1'

# Claude Code 에 물어보기 ·
#   「Q 의 낱말 중 GOLD 문서 본문에 그대로 있는 것을 찾아 줘」

TF-IDF 로 문서와 질문을 벡터로 바꾸고, 질문과 겹치는 낱말이 많은 문서를 찾습니다.
- **TF-IDF**: 낱말이 그 문서에 자주 나오고(TF) 다른 문서에는 드물수록(IDF) 큰 점수를 주는 방식입니다. 문서마다 낱말별 점수 벡터가 생깁니다.
- `fit_transform(docs)` 는 문서에서 낱말 사전을 만들고 문서×낱말 점수 행렬 `doc_word_scores` 를 만듭니다. `transform([Q])` 는 질문을 같은 사전으로 바꿉니다.
- 두 벡터를 곱하면(`@`) 겹치는 낱말의 점수만 더해집니다. 겹치는 낱말이 없으면 0점입니다.

출력에서 0점 문서 수와 정답 문서의 점수를 봅니다. 이어서 질문 낱말(「칠러가」)과 문서 낱말(「칠러」)이 낱말 목록에 있는지, 나온 문서 수와 정답 문서 점수를 봅니다.

In [ ]:
# 예제 3-2 (4) TF-IDF 로 키워드 검색을 합니다
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')   # 낱말 단위로 자릅니다
doc_word_scores = tfidf.fit_transform(docs)             # 문서 × 낱말 점수 행렬
# 질문을 같은 낱말 사전으로 바꿔 문서마다 겹치는 낱말 점수를 더합니다
question_vector = tfidf.transform([Q])
scores = (doc_word_scores @ question_vector.T).toarray().ravel()
zero_count = (scores == 0).sum()
gold_doc_score = scores[names.index(GOLD)]
print('최고 점수', scores.max(), '· 0점 문서', zero_count,
      '· 정답 문서 점수', gold_doc_score)
# 문서 낱말과 질문 낱말이 낱말 목록에 있는지, 몇 문서에 나왔는지 봅니다
for word in ['칠러', '챔버', '기준', '온도', '칠러가', '챔버가', '기준이']:
    if word not in tfidf.vocabulary_:
        print(word, '· 낱말 목록에 없음')
        continue
    column = tfidf.vocabulary_[word]
    word_scores = doc_word_scores[:, column].toarray().ravel()
    doc_count = (word_scores > 0).sum()
    gold_word_score = word_scores[names.index(GOLD)]
    print(word, f'· 나온 문서 {doc_count}건 · 정답 점수 {gold_word_score:.3f}')

# Claude Code 에 물어보기 ·
#   「조사를 떼는 토크나이저로 바꿔 정답 점수를 다시 내 줘」

예제 3-2 (4) 의 키워드 점수 계산을 함수 `kw_search()` 로 묶어, 현장 말투 질문 다섯 개를 차례로 넣어 봅니다. 예제 3-3 (6) 에서 같은 다섯 질문을 임베딩으로 다시 찾습니다.
- `scores.argsort()[::-1][:k]` 는 점수가 큰 순서로 k 개 문서 번호를 돌려줍니다. 점수가 0 이면 그 순서는 의미가 없습니다.

정답 문서는 예제 3-3 (6) 과 같은 목록(`KGOLD`)입니다.

출력에서 질문마다 1위 점수가 0 인지, 1위 문서가 정답인지 봅니다. 마지막 줄의 질문을 각자 바꿔 실행해 봅니다.

In [ ]:
# 예제 3-2 (5) 키워드 검색을 함수로 만들어 질문 다섯 개를 넣어 봅니다
def kw_search(q, k=3):
    """낱말이 겹치는 점수 순으로 상위 k건의 (문서 이름, 점수)를 돌려줍니다."""
    question_vector = tfidf.transform([q])
    # 문서마다 겹치는 낱말 점수를 더합니다
    scores = (doc_word_scores @ question_vector.T).toarray().ravel()
    top_indices = scores.argsort()[::-1][:k]        # 점수 큰 순서로 k개
    return [(names[i], round(float(scores[i]), 3)) for i in top_indices]

KQ = ['칠러가 안 도는 것 같은데 챔버가 멈추는 기준이 뭔가요',
      '에치 리크 체크는 어떻게 하나요',
      'CVD 알람이 뜨면 뭘 먼저 하나요',
      'He 배면압 설정 기준이 어떻게 되나요',
      '포토 웨이퍼를 수동으로 넣는 절차']
KGOLD = ['SOP-ETCH-007_v2.1', 'SOP-ETCH-006', 'SOP-CVD-009',
         'SOP-ETCH-012_v2.0', 'SOP-포토-013']      # 질문별 정답 문서
for question, gold_doc in zip(KQ, KGOLD):
    top_results = kw_search(question, k=1)
    if top_results[0][0] == gold_doc:
        verdict = '· 정답'
    else:
        verdict = f'· 오답(정답 {gold_doc})'
    print(question, '→', top_results, verdict)
print('각자 질문 →', kw_search('약품 다룰 때 보호구'))        # 질문을 바꿔 보세요

# Claude Code 에 물어보기 ·
#   「k=3 으로 바꿔 다섯 질문의 상위 3 안 정답 수를 세어 줘」

## 예제 3-3 · 임베딩과 벡터 유사도 검색
같은 질문을 임베딩(문장을 숫자 384개로 바꾼 벡터)으로 다시 찾고, 질문 다섯 개로 몇 위에 정답이 오는지 잽니다.

문장 하나를 e5 모델로 임베딩해 숫자 384개짜리 벡터로 바꿉니다.
- **임베딩**: 뜻을 숫자 벡터로 나타낸 것입니다. 뜻이 비슷한 문장은 벡터 방향이 비슷합니다.
- **normalize_embeddings=True**: 벡터를 길이 1 로 맞춥니다(정규화). 그러면 두 벡터의 곱이 곧 코사인 유사도가 됩니다.
- `'passage: '` 는 e5 모델에서 문서 쪽 문장 앞에 붙이는 접두어입니다. 질문 쪽에는 `'query: '` 를 붙입니다.
- `SentenceTransformer(...)` 는 문장을 벡터로 바꾸는 모델을 불러옵니다. 처음 실행할 때 한 번 약 470MB 를 내려받아 1~2분 걸립니다.

출력에서 모양 (1, 384), 길이 1.0, 앞 8개 숫자를 확인합니다.

In [ ]:
# 예제 3-3 (1) 임베딩 모델을 불러오고 문장 하나를 벡터로 바꿉니다 · 숫자 384개, 길이 1
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('intfloat/multilingual-e5-small')   # 처음 한 번 470MB
# passage: 는 문서 쪽 문장에 붙이는 접두어입니다
vectors = model.encode(['passage: ESC 온도 상한은 62.0°C'],
                       normalize_embeddings=True)
length = round(float((vectors[0] ** 2).sum()), 4)   # 벡터 길이(제곱합)를 셉니다
print(vectors.shape, '· 길이', length)
print(vectors[0][:8].round(3))

# Claude Code 에 물어보기 ·
#   「두 문장의 벡터를 만들어 코사인 유사도를 계산해 줘」

문서 쪽 문장 다섯 개와 질문 두 개를 임베딩해, 질문마다 다섯 문장을 유사도 순으로 줄 세웁니다. 질문과 정답 문장에는 겹치는 낱말이 없습니다.
- **코사인 유사도**: 두 벡터의 방향이 얼마나 같은지 나타내는 값입니다. 1 에 가까울수록 뜻이 가깝고, 길이 1 벡터끼리는 `a @ b` 가 곧 이 값입니다.
- `passage_vectors @ question_vector` 은 다섯 문장 각각과 질문의 유사도를 한 번에 냅니다. `argsort()[::-1]` 은 큰 점수부터 차례로 번호를 돌려줍니다.

출력에서 낱말이 겹치지 않는데도 뜻이 맞는 문장이 1위에 오는지 봅니다. 점수는 0.7~0.9 사이에 모여 있어 절댓값보다 순서로 읽습니다.

In [ ]:
# 예제 3-3 (2) 뜻이 가까운 문장이 1위에 오는지 봅니다 · 다섯 문장을 줄 세웁니다
P = ['챔버 온도가 상한을 넘으면 즉시 공정을 중지하고 인터락을 건다',
     '클린룸 방진복은 모자, 마스크, 상의, 하의, 장갑 순서로 입는다',
     '연차는 분기마다 사용 계획을 제출한다',
     '웨이퍼 반송 로봇은 2주마다 그리스를 바른다',
     '구내식당 저녁 배식은 18시에 시작한다']
passage_vectors = model.encode(['passage: ' + passage for passage in P],
                               normalize_embeddings=True)
for question in ['설비가 과열되면 어떻게 하나요', '휴가 신청은 어떻게 하나요']:
    question_vector = model.encode(['query: ' + question],
                                   normalize_embeddings=True)[0]
    scores = passage_vectors @ question_vector    # 길이 1 벡터끼리 곱 = 코사인 유사도
    print('질문:', question)
    for i in scores.argsort()[::-1]:              # 점수 큰 순서로
        print(f'   {scores[i]:.3f}  {P[i]}')

# Claude Code 에 물어보기 ·
#   「P 에 문장을 두 개 더 넣고 순위가 바뀌는지 돌려 줘」

문서 253건을 모두 임베딩해 행렬 `E` 를 만듭니다. 이 행렬이 검색 인덱스이며, 문서마다 행 하나에 숫자 384개가 들어갑니다.
- `batch_size=32` 는 32건씩 묶어 계산한다는 뜻입니다. `E.nbytes` 는 행렬이 차지하는 메모리 바이트 수입니다.

출력에서 행렬 모양, 크기(MB), 걸린 시간을 확인합니다.

In [ ]:
# 예제 3-3 (3) 문서 253건을 전부 벡터로 바꿔 인덱스를 만듭니다
started = time.time()
E = model.encode(['passage: ' + doc_text for doc_text in docs],
                 normalize_embeddings=True, batch_size=32)
seconds = time.time() - started
size_mb = round(E.nbytes / 1e6, 3)
print(E.shape, '·', size_mb, 'MB ·', f'{seconds:.1f}초')

# Claude Code 에 물어보기 ·
#   「E 를 파일로 저장하고 다시 읽는 코드를 써 줘」

예제 3-2 의 질문 `Q` 를 임베딩해 모든 문서와의 유사도를 한 번에 계산합니다.
- `E @ question_vector` 는 문서마다 질문과의 유사도를 한 번에 냅니다.
- **argsort**: 값을 작은 순으로 정렬했을 때의 번호(인덱스)를 돌려줍니다. `[::-1]` 로 뒤집으면 큰 순이 되고, `[:4]` 는 앞 4개입니다.

출력의 상위 4건에 정답 문서 `SOP-ETCH-007_v2.1` 이 있는지 보고, 예제 3-2 (4) 의 키워드 검색과 비교합니다.

In [ ]:
# 예제 3-3 (4) 같은 질문을 임베딩으로 다시 찾습니다
question_vector = model.encode(['query: ' + Q], normalize_embeddings=True)[0]
scores = E @ question_vector                 # 문서마다 질문과의 유사도
for i in scores.argsort()[::-1][:4]:         # 점수 큰 순서로 4개
    print(round(float(scores[i]), 4), names[i])

# Claude Code 에 물어보기 ·
#   「상위 4건 중 2~4위 문서의 첫 줄을 보여 줘」

질문 앞의 `'query: '` 접두어를 빼고 임베딩했을 때와 붙였을 때의 1위 문서와 점수를 봅니다. e5 모델은 질문에 `query:`, 문서에 `passage:` 를 붙인 데이터로 학습해, 접두어를 맞춰야 설계대로 동작합니다.

출력에서 두 경우의 1위 문서가 같은지 봅니다. 이 데이터에서는 점수 차이가 작지만, 점수 절댓값은 비교 기준이 아니므로 모델 설명서의 규칙대로 `query:` 를 붙입니다.

In [ ]:
# 예제 3-3 (5) query: 접두어를 빼면 점수가 어떻게 되나
plain_vector = model.encode([Q], normalize_embeddings=True)[0]
query_vector = model.encode(['query: ' + Q], normalize_embeddings=True)[0]
plain_scores = E @ plain_vector
query_scores = E @ query_vector
# 두 경우의 1위 문서와 점수를 나란히 봅니다
plain_top = names[int(plain_scores.argmax())]
query_top = names[int(query_scores.argmax())]
print('접두어 없이  ', round(float(plain_scores.max()), 4), plain_top)
print('query: 붙여서', round(float(query_scores.max()), 4), query_top)

# Claude Code 에 물어보기 ·
#   「다섯 질문 전부에서 접두어 유무로 1위가 바뀌는지 돌려 줘」

질문 다섯 개와 각 정답 문서로 검색 성능을 잽니다. 질문마다 정답 문서가 몇 위에 오는지 구합니다.
- `order.index(...) + 1` 은 정렬된 번호 목록에서 정답 문서의 자리를 찾고, 0부터 세므로 1을 더해 순위로 바꿉니다.
- **recall@k**: 정답 문서가 상위 k건 안에 든 질문의 비율입니다. recall@3 이 100% 면 상위 3건만 붙여도 모든 질문에 정답 문서가 들어갑니다.

정답이 1위가 아니면 앞에 온 문서도 함께 찍습니다.

출력에서 질문별 순위와 recall@1·recall@3 을 확인합니다.

In [ ]:
# 예제 3-3 (6) 질문 다섯 개로 정답 문서가 몇 위에 오는지 잽니다
QSET = [
    ('칠러가 안 도는 것 같은데 챔버가 멈추는 기준이 뭔가요', 'SOP-ETCH-007_v2.1'),
    ('에치 리크 체크는 어떻게 하나요',                      'SOP-ETCH-006'),
    ('CVD 알람이 뜨면 뭘 먼저 하나요',                      'SOP-CVD-009'),
    ('He 배면압 설정 기준이 어떻게 되나요',                  'SOP-ETCH-012_v2.0'),
    ('포토 웨이퍼를 수동으로 넣는 절차',                     'SOP-포토-013'),
]

def recall_at(ranks, k):
    """순위 목록에서 정답이 상위 k위 안에 든 질문의 비율(%)을 돌려줍니다."""
    hits = sum(rank <= k for rank in ranks)
    return hits / len(ranks) * 100

# 질문마다 정답 문서의 순위를 구합니다
ranks = []
for question, gold_doc in QSET:
    question_vector = model.encode(['query: ' + question],
                                   normalize_embeddings=True)[0]
    order = list((E @ question_vector).argsort()[::-1])   # 점수 큰 순서의 문서 번호
    rank = order.index(names.index(gold_doc)) + 1          # 0부터 세므로 +1 해서 순위
    ranks.append(rank)
    ahead = [names[i] for i in order[:rank - 1]]           # 정답보다 앞에 온 문서
    line = f'{gold_doc:20s} {rank}위'
    if ahead:
        line += f' · 앞에 온 문서 {ahead}'
    print(line)
print('recall@1', recall_at(ranks, 1), '% · recall@3', recall_at(ranks, 3), '%')

# Claude Code 에 물어보기 ·
#   「v1.2 가 앞에 온 질문에 버전을 명시하면 순위가 바뀌는지 보여 줘」

질문을 넣으면 상위 k건의 (문서 이름, 점수) 를 돌려주는 `search()` 를 만듭니다. 미션 3 에서 이 함수를 그대로 씁니다.
- `round(float(...), 3)` 은 numpy 숫자를 파이썬 실수로 바꿔 소수 셋째 자리까지 남깁니다.

질문을 바꿔 실행해 보고, 출력의 1위 문서 이름이 질문 내용과 맞는지 봅니다.

In [ ]:
# 예제 3-3 (7) 검색을 함수 하나로 만들고 각자 질문을 넣어 봅니다
def search(q, k=3):
    """유사도가 높은 순으로 상위 k건의 (문서 이름, 점수)를 돌려줍니다."""
    question_vector = model.encode(['query: ' + q], normalize_embeddings=True)[0]
    scores = E @ question_vector
    top_indices = scores.argsort()[::-1][:k]        # 점수 큰 순서로 k개
    return [(names[i], round(float(scores[i]), 3)) for i in top_indices]

# 질문을 바꿔 보세요
for question in ['고전압 작업할 때 잠금은 어떻게 하나요', '약품 다룰 때 보호구']:
    print(question, '→', search(question))

# Claude Code 에 물어보기 ·
#   「search 에 점수 하한을 두고 그 아래면 빈 목록을 주게 고쳐 줘」

예제 3-1 의 질문 `Q0` 로 검색해 1위 문서 한 건만 붙여 답을 받습니다. 검색으로 고른 근거를 붙여 답을 생성하는 방식이 RAG(검색 증강 생성)이며, 3일차에 본격적으로 만듭니다.
- **argmax**: 가장 큰 값의 번호를 돌려줍니다. 1위 문서 하나만 필요할 때 argsort 대신 씁니다.

출력에서 입력 토큰을 예제 3-1 (4) 의 전부 붙였을 때와 비교하고, 답이 문서의 수치와 맞는지 봅니다.

In [ ]:
# 예제 3-3 (8) 찾은 문서 한 건만 붙여 답하게 합니다 · 3일차에 만들 RAG 의 맛보기
import numpy as np                    # 숫자 배열 계산 라이브러리
# 질문과 가장 가까운 문서 한 건의 이름을 고릅니다
question_vector = model.encode(['query: ' + Q0], normalize_embeddings=True)[0]
top_doc = names[int(np.argmax(E @ question_vector))]
# 그 문서 한 건만 붙여 묻습니다
top_text = docs[names.index(top_doc)]
prompt = f'다음 문서를 근거로 답해줘.\n\n{top_text}\n\n질문: {Q0}'
answer, usage = ask_text(prompt)
print('검색 1위', top_doc, '· 입력 토큰', usage.input_tokens)
print(answer)

# Claude Code 에 물어보기 ·
#   「상위 3건을 붙이도록 고쳐 입력 토큰과 답을 비교해 줘」

검색 1위 문서가 정말 질문에 답하는 문서인지는 문서를 열어 읽어야 압니다. 질문 두 개로 `search()` 1위 문서의 앞부분을 열어 봅니다. 실습 미션 3 에서 질문 6개로 같은 판정을 합니다.
- `docs[names.index(top_doc)]` 은 문서 이름으로 본문을 찾습니다. `[:300]` 은 앞 300자입니다.

출력을 읽고 질문마다 「이 문서에 답이 있다 O / 없다 X」를 정하고, O 이면 근거가 되는 문장을 하나 고릅니다.

In [ ]:
# 예제 3-3 (9) 1위 문서를 열어 질문에 답하는 문서인지 읽어 봅니다 · 미션 3 판정 연습
questions = ['고전압 작업할 때 잠금은 어떻게 하나요',
             '포토 웨이퍼를 수동으로 넣는 절차']        # 질문을 바꿔 보세요
for question in questions:
    top_doc, score = search(question, k=1)[0]
    print(f'질문: {question}\n1위: {top_doc} ({score})')
    preview = docs[names.index(top_doc)][:300].replace('\n', ' ')   # 앞 300자를 한 줄로
    print(preview)
    print('-' * 60)

# Claude Code 에 물어보기 ·
#   「1위 문서에서 질문의 답이 되는 문장만 뽑아 줘」

같은 뜻을 세 가지 말로 묻고 1위 문서가 그대로인지 봅니다. 현장 말 · 문서 말 · 약어를 섞었습니다. 키워드 검색이라면 낱말이 달라 결과가 크게 바뀌는 자리입니다.

출력에서 세 질문의 1위가 같은 문서인지, 1위와 2위 점수가 얼마나 벌어지는지 봅니다.

In [ ]:
# 예제 3-3 (10) 같은 뜻을 다른 말로 물어 1위가 그대로인지 봅니다
questions = ['칠러가 안 도는 것 같은데 챔버가 멈추는 기준이 뭔가요',   # 현장 말
             '챔버 인터락이 자동 작동하는 온도 조건',                 # 문서 말
             'ESC 과열 시 인터락 기준']                              # 약어
for question in questions:
    print(question, '→', search(question, k=2))

# Claude Code 에 물어보기 ·
#   「영어로 같은 질문을 물어 1위가 유지되는지 돌려 줘」

예제 3-1 (3) 에서 잰 「문서 253건 전부 붙이기」 입력 토큰 `full_input_tokens` 과, 예제 3-3 (8) 에서 받은 「한 건만 붙이기」 입력 토큰 `usage.input_tokens` 를 같은 정가를 곱해 요금으로 바꿉니다. 파트 1 에서 계산한 토큰 요금과 같은 식입니다.
- 정가는 예제 3-1 (3) 과 같이 입력 100만 토큰당 2달러로 둡니다. `CALLS` 를 바꿔 하루 질문 수에 따라 어떻게 달라지는지 봅니다.

출력에서 두 방식의 질문 한 번 요금과 하루 요금, 몇 배 차이인지 확인합니다.

In [ ]:
# 예제 3-3 (11) 전부 붙이기와 한 건 붙이기의 요금을 나란히 계산합니다
P_IN = 2 / 1e6          # 입력 토큰 하나의 요금(달러) · 100만 토큰당 2달러
CALLS = 1000            # 하루 질문 수 · 바꿔 보세요
cases = [('전부 붙이기', full_input_tokens),     # 예제 3-1 (3) 에서 센 값
         ('한 건 붙이기', usage.input_tokens)]   # 예제 3-3 (8) 에서 받은 값
for label, tokens in cases:
    cost_once = tokens * P_IN
    cost_daily = tokens * P_IN * CALLS
    print(f'{label:7s} 입력 {tokens:7d} 토큰 · 한 번 ${cost_once:.4f} · '
          f'하루 {CALLS}번 ${cost_daily:.2f}')
print(f'차이 {full_input_tokens / usage.input_tokens:.0f}배')

# Claude Code 에 물어보기 ·
#   「상위 3건 붙이기 줄을 cases 에 더해 계산해 줘」

## 마치며
오늘 만든 파일: `문의_분류결과.xlsx` · `inquiries.db` · `run_meta.json`
오늘 만든 인덱스 `E` 와 `names` 는 3일차 첫 셀에서 같은 코드로 다시 만듭니다.
실습 미션은 `2일차_실습.ipynb` 에서 합니다.